## 기본 .env api 연결 확인 

In [9]:
# 필요한 패키지 로드 및 환경변수 설정
import os
from dotenv import load_dotenv

load_dotenv()

# API 키 확인 (출력값에 키가 정상적으로 나오는지 확인)
print("OPENAI_KEY:", "설정됨" if os.getenv("OPENAI_API_KEY") else "없음")
print("GROQ_KEY:", "설정됨" if os.getenv("GROQ_API_KEY") else "없음")
print("GOOGLE_KEY:", "설정됨" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_KEY: 설정됨
GROQ_KEY: 설정됨
GOOGLE_KEY: 설정됨


### 문제 1-1 : 기본 Chain 만들기 - AI 요리사

In [14]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_upstage import ChatUpstage
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 환경변수 로드
load_dotenv()

# 2. 핵심 API 제공사 동적 선택 팩토리 함수
def get_llm(provider: str = "upstage", temperature: float = 0.3):
    """provider 이름에 따라 알맞은 LLM 인스턴스를 동적으로 반환합니다."""
    provider = provider.lower()
    
    if provider == "openai":
        return ChatOpenAI(
            api_key=os.getenv("OPENAI_API_KEY"), 
            model="gpt-4o-mini", 
            temperature=temperature
        )
    elif provider == "gemini":
        return ChatGoogleGenerativeAI(
            google_api_key=os.getenv("GOOGLE_API_KEY"), 
            model="gemini-1.5-flash", 
            temperature=temperature
        )
    elif provider == "upstage":
        return ChatUpstage(
            api_key=os.getenv("UPSTAGE_API_KEY"), 
            model="solar-1-mini-chat", 
            temperature=temperature
        )
    elif provider == "groq":
        return ChatGroq(
            api_key=os.getenv("GROQ_API_KEY"), 
            model="llama-3.3-70b-versatile", 
            temperature=temperature
        )
    else:
        raise ValueError(f"지원하지 않는 프로바이더입니다: {provider}")


# 3. 프로바이더 인수를 받아 실행하는 레시피 생성 함수
def test_recipe_with_provider(ingredients: str, provider: str = "upstage") -> str:
    print(f"\n🔄 [{provider.upper()}] 모델로 요청을 전송합니다...")
    
    # 동적으로 LLM 선택
    llm = get_llm(provider=provider)
    
    # 프롬프트 및 체인 구성
    prompt = PromptTemplate.from_template(
        "다음 재료들로 만들 수 있는 요리를 추천하고, 간단한 레시피를 알려주세요.\n재료: {ingredients}"
    )
    chain = prompt | llm | StrOutputParser()
    
    # 실행 및 결과 반환
    return chain.invoke({"ingredients": ingredients})


# 4. 프로바이더를 변경해가며 테스트 실행
# (현재 정상 작동이 확인된 "upstage" 외에 다른 프로바이더도 인수를 바꿔가며 테스트해볼 수 있습니다)
providers_to_test = ["gemini"] # 필요시 ["upstage", "openai", "gemini", "groq"] 등으로 확장 가능

for prov in providers_to_test:
    try:
        result = test_recipe_with_provider("계란, 밥, 김치", provider=prov)
        print(f"=== [{prov.upper()} 결과 성공] ===")
        print(result)
    except Exception as e:
        print(f"❌ [{prov.upper()} 오류 발생]: {e}")


🔄 [GEMINI] 모델로 요청을 전송합니다...
❌ [GEMINI 오류 발생]: Error calling model 'gemini-1.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


### 문제 1-2 : 2단계 Multi Chain 만들기 - 여행지 정보 시스템

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1단계: 명소 추천 프롬프트 및 체인
prompt1 = ChatPromptTemplate.from_messages([
    ("system", "당신은 전문 여행 가이드입니다. 사용자가 입력한 도시를 대표하는 가장 유명한 명소 1곳의 '이름'만 짧게 대답하세요."),
    ("user", "{location}")
])
chain1 = prompt1 | llm | StrOutputParser()

# 2단계: 상세 정보 생성 프롬프트 및 체인
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 여행 가이드입니다. 주어진 명소에 대해 역사, 특징, 방문 팁을 각각 번호 매겨 상세히 설명해주세요."),
    ("user", "{landmark}에 대한 상세 정보를 알려주세요.")
])
chain2 = prompt2 | llm | StrOutputParser()

# 다중 체인(Runnable Parallel/Assign) 연결
tour_chain = (
    {"landmark": chain1} 
    | RunnablePassthrough.assign(details=chain2)
)

# 실행
tour_result = tour_chain.invoke({"location": "로마"})
print("=== [여행지 다중 체인 결과] ===")
print(f"선택된 명소: {tour_result['landmark']}")
print(f"상세 정보:\n{tour_result['details']}")

=== [여행지 다중 체인 결과] ===
선택된 명소: 콜로세움
상세 정보:
물론입니다! 콜로세움은 로마에서 가장 유명한 명소 중 하나로, 그 역사적인 중요성과 건축적 아름다움으로 많은 관광객들을 끌어들입니다. 다음은 콜로세움에 대한 상세 정보입니다.

### 1. 역사
- **건설 시기**: 콜로세움은 70-80년 경, 로마 제국 시대에 건설되었습니다.
- **건설 목적**: 주로 검투사 경기와 다양한 공공 오락 행사를 위해 사용되었습니다.
- **건설자**: 베스파시아누스 황제가 건설을 시작했고, 그의 아들 티투스 황제가 완성했습니다.
- **명칭의 유래**: 원래는 "플라비우스 원형극장"이라 불렸으나, 근처에 있던 네로 황제의 거대한 동상(콜로서스) 때문에 "콜로세움"으로 알려지게 되었습니다.

### 2. 특징
- **규모**: 콜로세움은 길이 189m, 너비 156m, 높이 48m로, 최대 8만 명의 관중을 수용할 수 있었습니다.
- **구조**: 주로 트라베르틴 석회암과 콘크리트로 만들어졌으며, 4층으로 구성되어 있습니다. 각 층은 아치로 장식되어 있습니다.
- **하부 구조**: 복잡한 기계 장치와 함께, "벨라리움"이라는 덮개를 통해 날씨에 따라 경기장을 보호할 수 있었습니다.
- **입구**: 80개의 아치형 입구가 있어, 관중들이 빠르게 입장하고 퇴장할 수 있었습니다.

### 3. 방문 팁
- **입장권 예약**: 콜로세움은 매우 인기 있는 명소이므로, 가능하면 온라인으로 미리 입장권을 예약하는 것이 좋습니다.
- **방문 시간**: 아침 일찍이나 오후 늦게 방문하는 것이 좋습니다. 이 시간대에는 줄이 짧고, 더위를 피할 수 있습니다.
- **가이드 투어**: 전문 가이드와 함께하는 투어를 통해 콜로세움의 역사와 건축에 대한 깊은 이해를 얻을 수 있습니다.
- **주변 명소**: 콜로세움 근처에는 로마 포럼과 팔라티노 언덕이 있어, 함께 방문하는 것이 좋습니다.
- **사진 촬영**: 콜로세움의 전체 모습을 담기 위해서는 주변 공원이나 높은

### 문제 1-3 : FewShotPromptTemplate과 시스템 메시지 활용 

In [16]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate

# 퓨샷 예시 데이터
examples = [
    {"news": "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다.", "keywords": "삼성전자, 인공지능, 출시"},
    {"news": "한국은행이 오늘 기준금리를 연 3.5%로 동결했습니다.", "keywords": "한국은행, 기준금리, 동결"}
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "뉴스 키워드 추출 전문가입니다. 기사 내용을 읽고 가장 핵심적인 키워드 3개를 추출하세요."),
    few_shot_prompt,
    ("human", "{input}")
])

news_chain = final_prompt | llm | StrOutputParser()

# 실행
news_text = "현대자동차가 미국 전기차 공장 가동 시기를 앞당기고 공격적인 현지 마케팅을 펼치겠다고 발표했다."
news_result = news_chain.invoke({"input": news_text})
print("=== [뉴스 키워드 추출 결과] ===")
print(news_result)

=== [뉴스 키워드 추출 결과] ===
키워드: 현대자동차, 미국, 전기차 공장
